#### 16 — Final Comparison and Conclusion

##### 1. Purpose

This notebook compares all major NLP approaches developed in the
support-ticket classification project and summarizes the lessons
learned across the full lifecycle.

The approaches compared are:

1. TF-IDF + Logistic Regression
2. MiniLM Embeddings + Logistic Regression
3. MiniLM Embeddings + Neural Network
4. Fine-tuned MiniLM Transformer

The notebook compares:

- test performance
- representation type
- classifier complexity
- semantic capability
- training cost
- inference cost
- deployment complexity
- strengths and limitations
- appropriate use cases

The notebook also summarizes the complete production workflow from
raw support-ticket data through Databricks Model Serving.

##### 2. Technologies

- PySpark
- pandas
- scikit-learn
- PyTorch
- Hugging Face Transformers
- Sentence Transformers
- MLflow
- Unity Catalog
- Databricks Model Serving

##### 3. Learning Progression

``` text

Raw Text
   ↓
Cleaning and Tokenization
   ↓
Vocabulary
   ↓
Bag of Words
   ↓
TF-IDF
   ↓
Traditional Machine Learning
   ↓
Pretrained Embeddings
   ↓
Neural Networks
   ↓
Transformer Foundations
   ↓
Transformer Fine-Tuning
   ↓
MLflow Tracking
   ↓
Unity Catalog Registration
   ↓
Databricks Model Serving
   ↓
Endpoint Testing

```

This progression is important because each stage addressed a limitation of the previous one:

Bag of Words → captures word occurrence

- TF-IDF → adds word importance
- Embeddings → add semantic meaning
- Neural Network → learns nonlinear decision boundaries
- Transformer fine-tuning → adapts contextual representations directly to the classification task

##### 4. Final model comparison

In [0]:
import pandas as pd

comparison_df = pd.DataFrame(
    {
        "approach": [
            "TF-IDF + Logistic Regression",
            "MiniLM Embeddings + Logistic Regression",
            "MiniLM Embeddings + Neural Network",
            "Fine-tuned MiniLM Transformer",
        ],
        "test_accuracy": [
            0.8462,
            1.0000,
            1.0000,
            1.0000,
        ],
        "macro_f1": [
            0.7083,
            1.0000,
            1.0000,
            1.0000,
        ],
        "weighted_f1": [
            0.7821,
            1.0000,
            1.0000,
            1.0000,
        ],
        "test_errors": [
            2,
            0,
            0,
            0,
        ],
    }
)

display(
    comparison_df
)

- The dataset is very small, especially the held-out test split.
- Therefore, perfect test performance should not be interpreted as proof that the models will achieve perfect generalization in production.
- The comparison is primarily valuable for understanding architectural
differences and model behavior.

##### 5. Representation comparison

In [0]:
representation_df = pd.DataFrame(
    {
        "approach": [
            "TF-IDF + Logistic Regression",
            "MiniLM Embeddings + Logistic Regression",
            "MiniLM Embeddings + Neural Network",
            "Fine-tuned MiniLM Transformer",
        ],
        "representation": [
            "Sparse TF-IDF vector",
            "Dense MiniLM sentence embedding",
            "Dense MiniLM sentence embedding",
            "Task-adapted contextual Transformer representation",
        ],
        "dimension": [
            85,
            384,
            384,
            "384 hidden dimensions per token",
        ],
        "representation_learned_during_task_training": [
            "No",
            "No",
            "No",
            "Yes",
        ],
        "classifier": [
            "Logistic Regression",
            "Logistic Regression",
            "Neural Network",
            "Linear classification head",
        ],
        "semantic_awareness": [
            "Low",
            "High",
            "High",
            "High + task-adapted",
        ],
    }
)

display(
    representation_df
)

- TF-IDF --> "refund" and "money back" may look unrelated
- MiniLM embedding --> understands semantic similarity
- Fine-tuned Transformer --> can additionally adapt those semantic representations toward our support-ticket labels

##### 6. Classifier comparison

In [0]:
classifier_df = pd.DataFrame(
    {
        "model": [
            "Logistic Regression",
            "Neural Network",
            "Transformer Classifier",
        ],
        "decision_function": [
            "Linear",
            "Nonlinear",
            "Deep contextual + classification head",
        ],
        "trainable_complexity": [
            "Low",
            "Medium",
            "High",
        ],
        "interpretability": [
            "Higher",
            "Moderate",
            "Lower",
        ],
        "compute_requirement": [
            "Low",
            "Moderate",
            "Higher",
        ],
    }
)

display(
    classifier_df
)

In [0]:
classifier_df = pd.DataFrame(
    {
        "approach": [
            "TF-IDF + Logistic Regression",
            "MiniLM Embeddings + Logistic Regression",
            "MiniLM Embeddings + Neural Network",
            "Fine-tuned MiniLM Transformer",
        ],

        "classifier": [
            "Logistic Regression",
            "Logistic Regression",
            "Neural Network",
            "Linear classification head",
        ],

        "decision_boundary": [
            "Linear",
            "Linear",
            "Nonlinear",
            "Deep contextual + linear head",
        ],

        "classifier_architecture": [
            "85 → 4",
            "384 → 4",
            "384 → 64 → 32 → 4",
            "384 → 4",
        ],

        "representation_updated_during_training": [
            "No",
            "No",
            "No",
            "Yes",
        ],

        "complexity": [
            "Low",
            "Low",
            "Medium",
            "High",
        ],
    }
)

display(
    classifier_df
)

```  text

Logistic Regression
    y = softmax(Wx + b)

Neural Network
    x
    ↓
Linear
    ↓
ReLU
    ↓
Linear
    ↓
ReLU
    ↓
Linear
    ↓
class logits

Transformer
    raw tokens
    ↓
many attention + FFN layers
    ↓
contextual representation
    ↓
classification head
    ↓
class logits

```

##### 7. Parameter complexity

In [0]:
#For the models where we measured parameter counts:

parameter_df = pd.DataFrame(
    {
        "model": [
            "Embedding + Logistic Regression",
            "Embedding + Neural Network",
            "Fine-tuned Transformer",
        ],
        "trainable_parameters": [
            1540,
            26852,
            22714756,
        ],
    }
)

display(
    parameter_df
)

In [0]:
parameter_df = pd.DataFrame(
    {
        "approach": [
            "TF-IDF + Logistic Regression",
            "MiniLM Embeddings + Logistic Regression",
            "MiniLM Embeddings + Neural Network",
            "Fine-tuned MiniLM Transformer",
        ],
        "task_trainable_parameters": [
            344,
            1540,
            26852,
            22714756,
        ],
        "representation_model": [
            "TF-IDF",
            "MiniLM",
            "MiniLM",
            "MiniLM Transformer",
        ],
        "representation_updated": [
            "TF-IDF fitted on training data",
            "No - MiniLM frozen",
            "No - MiniLM frozen",
            "Yes - Transformer fine-tuned",
        ],
        "classifier": [
            "Logistic Regression",
            "Logistic Regression",
            "Neural Network",
            "Linear classification head",
        ],
    }
)

display(parameter_df)

##### This is one of the most important project lessons:

``` text

1,540 parameters
        ↓
same test accuracy

26,852 parameters
        ↓
same test accuracy

22.7 million parameters
        ↓
same test accuracy

```

More parameters do not automatically produce a better model.

The correct model depends on:

- dataset size
- semantic complexity
- latency requirements
- cost constraints
- maintainability
- interpretability
- expected production variation

##### 8. Why TF-IDF failed on the Login examples

Recall the two test errors:

account locked after many attempts

- were actual: Login
- but predicted: Cancellation

Why?

The test split contained words such as: locked, attempts, many

that were not present in the training vocabulary.

TF-IDF therefore could not use those unseen words as features.

The model mainly saw known features such as: account

and learned associations from the small training data.

This illustrates: TF-IDF depends heavily on training vocabulary

whereas pretrained embeddings already contain semantic knowledge learned from much larger external corpora.

## Why TF-IDF Failed on the Login Examples

The TF-IDF + Logistic Regression model made two errors on the held-out
test set. Both involved the Login category.

One problematic ticket was:

`account locked after many attempts`

The actual category was `Login`, while the model predicted
`Cancellation`.

Several informative words in the ticket, including `locked` and
`attempts`, were not present in the training vocabulary.

Because the TF-IDF vectorizer was correctly fitted only on the training
data, unseen test words did not receive TF-IDF features. Logistic
Regression therefore never received those words as input features.

This demonstrates an important separation between representation and
classification:

Raw text
    ↓
TF-IDF representation
    ↓
Logistic Regression classifier

The classifier can only learn from information preserved by its input
representation.

Pretrained MiniLM embeddings improved this limitation by representing
the semantic meaning of the complete ticket using knowledge learned
during large-scale pretraining.

This allowed semantically related expressions such as account lockout
and login problems to receive similar representations even when the
exact wording differed from the small project training dataset.

##### 9. Why embeddings helped

MiniLM transformed each ticket into: 384 semantic numbers

The representation was pretrained on external language data.

Therefore sentences such as: account locked after many attempts and unable to login because account is locked

can be close in embedding space even if the exact wording differs.

This is the key advantage:

TF-IDF
word matching

vs

Embeddings
semantic meaning

## Why Embeddings Helped

The next approach replaced the sparse TF-IDF representation with
pretrained MiniLM sentence embeddings.

Each support ticket was represented by a dense 384-dimensional vector.

Unlike TF-IDF dimensions, which correspond to vocabulary terms, the
MiniLM dimensions are learned latent features. Semantic information is
distributed across the complete embedding vector.

MiniLM was pretrained on large external text datasets, allowing it to
represent semantic relationships that were not learnable from the
project's small training dataset alone.

For example, the project observed high cosine similarity between:

`account locked after many attempts`

and

`unable to login because account is locked`

even though the wording was not identical.

An important experiment was that the classifier remained Logistic
Regression while only the representation changed:

TF-IDF
    ↓
Logistic Regression
    ↓
Test Accuracy = 0.8462

MiniLM Embeddings
    ↓
Logistic Regression
    ↓
Test Accuracy = 1.0000

This demonstrates the importance of feature representation.

A stronger semantic representation allowed a simple linear classifier
to separate the support-ticket categories successfully on the small
held-out test set.

Because the test set contained only 13 examples, the perfect result
should not be interpreted as evidence of perfect production
generalization.

Latent simply means hidden or not directly named/observed.

TF-IDF feature
→ explicit / interpretable
→ "this dimension represents the word refund"

Embedding feature
→ latent / learned
→ "this dimension is part of a learned semantic representation"


##### 10. Why the neural network did not outperform logistic regression

Both used the same: MiniLM 384-dimensional embeddings

Results:

Embedding + Logistic Regression
Accuracy = 1.0

Embedding + Neural Network
Accuracy = 1.0

The neural network added nonlinear capacity:

384 → 64 → 32 → 4

but the dataset was already separable enough in embedding space.

Therefore, Logistic Regression was sufficient.

This teaches:

strong representation
+
simple classifier
can outperform
weak representation
+
complex classifier

That is a very important practical ML principle.


The embedding-based Logistic Regression and Neural Network models used
the same pretrained MiniLM representation.

Each ticket was first converted into a fixed 384-dimensional sentence
embedding.

The primary difference was therefore the classifier.

Logistic Regression used a linear mapping:

384 → 4

The Neural Network used:

384 → 64 → ReLU → 32 → ReLU → 4

The ReLU activations allow the Neural Network to learn nonlinear
decision boundaries and therefore provide greater modeling capacity
than Logistic Regression.

However, both approaches achieved:

Test Accuracy = 1.0000
Macro F1 = 1.0000
Weighted F1 = 1.0000

The pretrained MiniLM embeddings already represented the four ticket
categories well enough for a simple linear classifier to separate the
small held-out test set successfully.

The additional nonlinear capacity of the Neural Network therefore
produced no measurable improvement on this dataset.

This demonstrates an important machine-learning principle:

A stronger or more complex classifier does not automatically produce
a better model.

When a strong pretrained representation makes the classes sufficiently
separable, a simple classifier may be the preferred solution because
it requires fewer trainable parameters and less training complexity.

##### 11. Why fine-tune the Transformer at all?

If frozen embeddings + Logistic Regression already achieved 100% on this tiny test set, why fine-tune?

Because the goal was not only to beat accuracy.

Fine-tuning teaches us how modern NLP systems adapt pretrained language models to downstream tasks.

The difference is:

Frozen Embeddings

``` text

Text
 ↓
pretrained MiniLM
 ↓
384 vector
 ↓
classifier

```

Transformer weights stay fixed

versus:

Fine-Tuning

``` text

Text
 ↓
MiniLM Transformer
 ↓
classification loss
 ↓
backpropagation
 ↓
Transformer weights update

```

So fine-tuning can adapt:

- attention patterns
- token representations
- internal Transformer layers

toward the support-ticket classification task.


The pretrained MiniLM embedding approach already achieved perfect
classification on the small held-out test set using Logistic
Regression.

However, the embedding model remained frozen.

In the frozen embedding approach:

Raw text
    ↓
Pretrained MiniLM
    ↓
Fixed 384-dimensional embedding
    ↓
Trainable classifier

Only the classifier learned from the support-ticket labels.

Full Transformer fine-tuning changes this learning process.

Raw text
    ↓
Tokenizer
    ↓
Trainable MiniLM Transformer
    ↓
Contextual token representations
    ↓
Masked mean pooling
    ↓
384-dimensional representation
    ↓
Trainable classification head

The classification loss is backpropagated through both the
classification head and the pretrained Transformer.

This allows the Transformer representation itself to adapt toward the
support-ticket classification task.

Fine-tuning is an example of transfer learning: knowledge learned
during large-scale language-model pretraining is reused and adapted
using a much smaller task-specific labeled dataset.

In this project, fine-tuning did not produce a measurable improvement
over frozen MiniLM embeddings because both approaches achieved perfect
classification on the 13-example test set.

The result should therefore not be interpreted as evidence that one
approach will always outperform the other.

The value of this stage was learning how a pretrained Transformer can
be adapted, evaluated, packaged, registered, and deployed for a
task-specific production NLP workflow.

##### 12. Frozen embeddings vs fine-tuning

In [0]:
fine_tuning_df = pd.DataFrame(
    {
        "attribute": [
            "Starting point",
            "Transformer parameters",
            "Representation",
            "Classifier parameters",
            "Task-trainable parameters",
            "Training cost",
            "Overfitting risk",
            "Task adaptation",
            "Best fit",
        ],
        "frozen_embeddings": [
            "Pretrained MiniLM",
            "Frozen",
            "Fixed pretrained 384-d embeddings",
            "Updated",
            "Low",
            "Lower",
            "Lower",
            "Classifier only",
            "Small datasets / strong baseline",
        ],
        "full_fine_tuning": [
            "Pretrained MiniLM",
            "Updated",
            "Task-adapted contextual representations",
            "Updated",
            "High",
            "Higher",
            "Higher",
            "Transformer + classifier",
            "When task-specific adaptation is justified",
        ],
    }
)

display(fine_tuning_df)

##### Frozen Embeddings vs. Full Fine-Tuning

Both approaches begin with a pretrained MiniLM model, but they differ
in which parameters are updated during task-specific training.

With frozen embeddings, MiniLM is used only as a pretrained feature
extractor. Each ticket is converted into a fixed 384-dimensional
embedding, and only the downstream classifier learns from the support
ticket labels.

With full fine-tuning, the classification loss is backpropagated
through the classification head and into the Transformer encoder.
The pretrained Transformer parameters are therefore updated along
with the classification head.

This allows the representation itself to adapt to the downstream task.

However, full fine-tuning introduces substantially more trainable
parameters, greater computational cost, and a higher risk of
overfitting when labeled data is limited.

For this project, frozen MiniLM embeddings already provided a strong
representation. Full fine-tuning was valuable for learning how
pretrained Transformer representations can be adapted to a specific
classification task and subsequently packaged and deployed.

##### 13. Production complexity comparison

In [0]:
production_df = pd.DataFrame(
    {
        "approach": [
            "TF-IDF + Logistic Regression",
            "MiniLM Embeddings + Logistic Regression",
            "MiniLM Embeddings + Neural Network",
            "Fine-tuned MiniLM Transformer",
        ],
        "task_training_cost": [
            "Very Low",
            "Low",
            "Moderate",
            "Higher",
        ],
        "inference_pipeline": [
            "TF-IDF + LR",
            "MiniLM + LR",
            "MiniLM + NN",
            "Fine-tuned MiniLM + Linear Head",
        ],
        "inference_cost": [
            "Very Low",
            "Moderate",
            "Moderate",
            "Higher",
        ],
        "deployment_complexity": [
            "Low",
            "Moderate",
            "Moderate",
            "Higher",
        ],
        "semantic_capability": [
            "Limited",
            "High",
            "High",
            "High + task-adapted",
        ],
    }
)

display(production_df)

The final Transformer required us to manage:

- model weights
- tokenizer
- classification head
- PyFunc wrapper
- Python dependencies
- project source code
- MLflow packaging
- Unity Catalog registration
- serving endpoint
- serving environment
- endpoint testing

That is much more infrastructure than:

TF-IDF
+
Logistic Regression

So production model selection is always a tradeoff.


Model complexity affects training and inference differently.

TF-IDF with Logistic Regression has both low training cost and low
inference cost because it does not require a neural language model.

The frozen MiniLM approaches require relatively little task-specific
training because the pretrained Transformer remains unchanged.
However, production inference still requires a MiniLM forward pass
to generate the 384-dimensional sentence embedding before the
classifier can make a prediction.

Therefore, the small number of trainable Logistic Regression
parameters should not be confused with the computational requirements
of the complete inference pipeline.

The Neural Network adds more classifier computation than Logistic
Regression, although the small classifier remains lightweight relative
to the MiniLM representation model.

Full Transformer fine-tuning has substantially greater training cost
because gradients must be computed through the pretrained Transformer
and millions of parameters are updated.

At inference time, however, the fine-tuned MiniLM model still performs
approximately the same type of Transformer forward computation as the
frozen MiniLM representation approaches.

Production model selection should therefore consider training cost,
end-to-end inference cost, latency, serving infrastructure, model
quality, and maintainability separately.

##### 14. Deployment lesson

This project gave us an especially valuable MLOps lesson.

The model passed:

training ✓

validation ✓

test evaluation ✓

MLflow reload ✓

Unity Catalog reload ✓

but initially failed: Databricks Serving ✗

because the serving runtime was separate from the notebook environment.

The final lesson:

A trained model
is not yet
a deployable model.

A deployable model needs:

``` text

model artifacts
+
dependencies
+
source code
+
input/output contract
+
serving-compatible runtime

```

A production deployment must package not only the trained model artifacts, but also the source code and Python dependencies required to reproduce inference in an isolated serving environment.


A model that works successfully in a development notebook is not
automatically ready for production deployment.

The Transformer model successfully completed training, validation,
test evaluation, MLflow logging, MLflow reload, Unity Catalog
registration, and registered-model inference.

However, the initial Databricks Model Serving deployment failed because
the serving environment is isolated from the Databricks Git repository.

The inference implementation depended on project source code under
`src/`. Although this code was available to the notebook environment,
it was not automatically available inside the Model Serving container.

The corrected MLflow model therefore packaged the required project
source code using `code_paths`.

A deployable ML model should contain everything required to reproduce
inference in an isolated environment:

Model artifacts
    +
Inference source code
    +
Python dependencies
    +
Input/output contract

The project also demonstrated the importance of separating training
dependencies from serving dependencies.

A development or training environment may contain many libraries used
for experimentation and model development. A production inference
environment should contain only the dependencies required to load the
model and execute predictions.

This improves deployment reproducibility, reliability, maintainability,
and operational efficiency.

##### 15. MLflow packaging lesson

The original environment inferred many unrelated packages:

- TensorFlow
- DeepSpeed
- XGBoost
- TorchVision
- ...

The better production principle is:

- Training environment --> can be large
- Serving environment --> should be minimal

For inference, we really needed approximately:

- MLflow
- PyTorch
- Transformers
- pandas
- NumPy
- model artifacts
- project inference code

This improves:

- build time
- reliability
- security
- maintainability
- deployment reproducibility

What exactly did MLflow need to package so our trained Transformer could become a self-contained inference model?

python_model = How should this model make a prediction?
artifacts — What trained files do I need? What model/data files does my inference code need?

artifacts={
    "transformer": transformer_dir,
    "tokenizer": tokenizer_dir,
    "classifier": classifier_path,
    "metadata": metadata_path,
}

transformer/
    ↓
fine-tuned Transformer weights + config

tokenizer/
    ↓
tokenizer vocabulary/configuration

classifier.pt
    ↓
trained classification-head weights

metadata.json
    ↓
class names and model metadata

code_paths = What application/source code must travel with the model?

artifacts → trained files/data needed by inference

code_paths → Python source code needed by inference

pip_requirements — What Python libraries do I need? What external Python libraries does the inference runtime require?

src/→ OUR Python code

torch / transformers / pandas → EXTERNAL Python packages

signature — What is the model contract? What shape/schema of data goes into and comes out of the model?



MLflow model packaging creates a reproducible inference contract rather
than storing only trained weights.

The production Transformer model required several different components.

`python_model`
defines how inference is performed. The custom MLflow PyFunc wrapper
loads the model components, validates raw input, performs tokenization,
runs Transformer inference, applies the classification head, and
returns prediction probabilities and categories.

`artifacts`
contain the trained files required during inference, including the
fine-tuned Transformer, tokenizer, classification-head weights, and
model metadata.

`code_paths`
package project-specific Python source code required by the inference
implementation. This was essential because the Databricks Model Serving
environment does not automatically contain the project's Git
repository.

`pip_requirements`
define the external Python libraries required to reconstruct the
inference environment.

`signature`
defines the expected model input and output schema.

`input_example`
provides a concrete example of a valid model request.

Together, these components create a portable MLflow model package:

Inference logic
    +
Model artifacts
    +
Project source code
    +
Python dependencies
    +
Input/output contract

This allows the model to be loaded outside the original development
notebook and executed inside an isolated production serving
environment.

python_model      → HOW inference works

artifacts         → WHAT trained files it needs

code_paths        → WHAT our source code it needs

pip_requirements  → WHAT external libraries it needs

signature         → WHAT data contract it expects

##### 16. Unity Catalog lifecycle

The production lifecycle now looks like:

``` text

MLflow Experiment
      ↓
contains
      ↓
MLflow Runs
      ↓
produces
      ↓
Logged Model
      ↓
registered into
      ↓
Unity Catalog Registered Model
      ↓
Model Version
      ↓
Candidate
      ↓
Databricks Serving Endpoint
      ↓
Endpoint Validation
      ↓
Champion

```

In [0]:
import mlflow
import pandas as pd

from mlflow import MlflowClient

from src.project_config import (
    CLASS_NAMES,
    TEXT_COL,
    MLFLOW_EXPERIMENT_NAME,
    MLFLOW_RUN_NAME,
    REGISTERED_MODEL_NAME,
    REGISTERED_MODEL_DESCRIPTION,
)

In [0]:
from mlflow import MlflowClient

from src.project_config import (
    REGISTERED_MODEL_NAME,
)

client = MlflowClient()

candidate = client.get_model_version_by_alias(
    name=REGISTERED_MODEL_NAME,
    alias="Candidate",
)

print("Candidate version:", candidate.version)

In [0]:
client.set_registered_model_alias(
    name=REGISTERED_MODEL_NAME,
    alias="Champion",
    version=candidate.version,
)

print(
    f"Promoted Version {candidate.version} "
    "to Champion."
)

In [0]:
champion = client.get_model_version_by_alias(
    name=REGISTERED_MODEL_NAME,
    alias="Champion",
)

print("Champion version:", champion.version)

assert champion.version == candidate.version

print("Champion alias verification passed.")

In [0]:
client.delete_registered_model_alias(
    name=REGISTERED_MODEL_NAME,
    alias="Candidate",
)

print(
    "Candidate alias removed after "
    "successful Champion promotion."
)


MLflow experiments organize training runs, while individual runs record
parameters, metrics, artifacts, and model outputs from specific
training executions.

A packaged MLflow model can be registered in Unity Catalog under a
long-lived registered-model name. Each registration creates a model
version, allowing multiple iterations of the same production model to
be governed independently.

Unity Catalog aliases provide logical lifecycle references to these
versions.

The `Candidate` alias identifies a model version currently undergoing
production validation.

The `Champion` alias identifies the validated model version selected
for production use.

The project followed the lifecycle:

Training Run
    ↓
Logged Model
    ↓
Registered Model Version
    ↓
Candidate
    ↓
Databricks Model Serving
    ↓
Live Endpoint Validation
    ↓
Champion

Aliases do not create copies of model artifacts. They provide logical
references to existing registered-model versions.

After successful endpoint validation, the Candidate version is promoted
to Champion.

The Candidate alias can then be removed so that it is available for a
future model version requiring evaluation.

This creates a repeatable lifecycle in which the current Champion can
remain in production while a future Candidate is independently
evaluated.

Candidate
= being evaluated

Champion
= approved production version

##### 17. Final Production Architecture


``` text

SUPPORT TICKET
    │
    ▼
Databricks Endpoint
    │
    ▼
MLflow PyFunc
    │
    ▼
BertTokenizerFast
    │
    ▼
Fine-tuned MiniLM
Transformer Encoder
    │
    ▼
Contextual Tokens
    │
    ▼
Masked Mean Pooling
    │
    ▼
384-dimensional vector
    │
    ▼
Classification Head
Linear(384,4)
    │
    ▼
    Logits
    │
    ▼
Softmax
    │
    ▼
Billing / Cancellation /
Login / Technical

```

The model was successfully deployed and tested through Databricks
Model Serving. After endpoint validation was completed, the serving
endpoint was deleted to avoid unnecessary ongoing serving cost.

The validated model remains governed in Unity Catalog.

##### 18. Final project comparison summary

In [0]:
final_summary_df = pd.DataFrame(
    {
        "approach": [
            "TF-IDF + Logistic Regression",
            "MiniLM Embeddings + Logistic Regression",
            "MiniLM Embeddings + Neural Network",
            "Fine-tuned MiniLM Transformer",
        ],
        "test_accuracy": [
            0.8462,
            1.0000,
            1.0000,
            1.0000,
        ],
        "macro_f1": [
            0.7083,
            1.0000,
            1.0000,
            1.0000,
        ],
        "representation": [
            "Sparse lexical TF-IDF",
            "Frozen semantic MiniLM embedding",
            "Frozen semantic MiniLM embedding",
            "Task-adapted contextual Transformer",
        ],
        "classifier": [
            "Logistic Regression",
            "Logistic Regression",
            "Neural Network",
            "Linear classification head",
        ],
        "task_trainable_parameters": [
            344,
            1540,
            26852,
            22714756,
        ],
        "complexity": [
            "Low",
            "Low",
            "Medium",
            "High",
        ],
        "main_strength": [
            "Simple and inexpensive baseline",
            "Strong semantic baseline",
            "Nonlinear classifier capacity",
            "Full task-specific representation adaptation",
        ],
    }
)

display(final_summary_df)


``` text

TF-IDF + LR
    ↓
simple but lexical limitations

MiniLM + LR
    ↓
major improvement from representation

MiniLM + NN
    ↓
more classifier capacity
but no measurable gain here

Fine-tuned Transformer
    ↓
most adaptable and complex
but no measurable gain on this tiny test set 

```
The largest observed improvement came from changing the representation from TF-IDF to pretrained semantic embeddings, not from increasing classifier complexity.

##### Key Learnings

1. NLP systems consist of both a representation layer and a classifier.

2. Bag of Words captures word occurrence but not semantic meaning.

3. TF-IDF improves lexical weighting but remains dependent on the
   training vocabulary.

4. Pretrained MiniLM embeddings provide dense semantic representations
   learned from large external corpora.

5. Strong pretrained representations can allow simple classifiers such
   as Logistic Regression to perform very well.

6. Neural Networks add nonlinear modeling capacity, but additional
   complexity does not guarantee improved performance.

7. Transformers create contextual representations using self-attention.

8. Full fine-tuning updates both the Transformer encoder and the
   classification head using task-specific loss.

9. Fine-tuning provides greater task adaptation but also increases
   training cost, trainable parameter count, and overfitting risk.

10. Trainable parameter count is different from total runtime
    computation and deployment complexity.

11. Model evaluation and endpoint testing validate different parts of
    the ML lifecycle.

12. A model that works in a notebook is not automatically deployable.

13. Production inference requires model artifacts, source code,
    Python dependencies, and a defined input/output contract.

14. MLflow provides reproducible model packaging and experiment
    tracking.

15. Unity Catalog provides governed model versioning and lifecycle
    aliases.

16. Production model selection should balance predictive quality,
    complexity, latency, cost, maintainability, and generalization.

##### Conclusion

This project progressed from traditional NLP representations to a
production-style Transformer classification workflow.

The learning path was:

``` text

Raw Text
    ↓
Cleaning and Tokenization
    ↓
Bag of Words
    ↓
TF-IDF
    ↓
Logistic Regression
    ↓
Pretrained MiniLM Embeddings
    ↓
Neural Network Classification
    ↓
Transformer Foundations
    ↓
Full Transformer Fine-Tuning
    ↓
MLflow Tracking and Packaging
    ↓
Unity Catalog Model Registration
    ↓
Databricks Model Serving
    ↓
Live Endpoint Testing

```

The TF-IDF baseline demonstrated the limitations of vocabulary-based
representations on unseen wording.

Replacing TF-IDF with pretrained MiniLM embeddings produced the largest
observed performance improvement in this project while retaining a
simple Logistic Regression classifier.

A Neural Network added nonlinear classification capacity but did not
produce a measurable improvement on the small held-out test set.

Full Transformer fine-tuning demonstrated how pretrained contextual
representations can be adapted to a task-specific classification
problem. Although it did not improve the already perfect test-set
metrics, it provided experience with modern NLP fine-tuning,
model packaging, dependency management, model governance, deployment,
and endpoint validation.

The project also demonstrated an important production lesson:
successful model training is only one part of an end-to-end ML system.

A deployable model must include everything required to reproduce
inference in an isolated environment, including:

- trained model artifacts
- inference source code
- runtime dependencies
- input/output contracts

The final Transformer model was successfully packaged with MLflow,
registered and governed through Unity Catalog, deployed through
Databricks Model Serving, and validated using live endpoint requests.

After successful live endpoint validation, the validated model version
was promoted from Candidate to Champion.

Unity Catalog now identifies the validated production model using the
`Champion` alias.

The Candidate alias was removed so that it can be assigned to a future
model version requiring evaluation.

Current lifecycle state:

Version 3 → Champion

After successful endpoint validation, the serving endpoint was deleted
to avoid unnecessary ongoing serving cost.

The validated registered model remains available in Unity Catalog for
future reuse, lifecycle promotion, or redeployment.

Overall, this project demonstrates an end-to-end production-style NLP
classification workflow in Databricks.